In [1]:
import pandas as pd
import numpy as np
import re
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("ColBERT-PRF Query Drift Analysis")
print("Analyzing topic drift through user selection patterns")
print("=" * 80)
print()

ColBERT-PRF Query Drift Analysis
Analyzing topic drift through user selection patterns



In [2]:
# ============================================================================
# SQL解析函数
# ============================================================================

def parse_sql_insert_statements(sql_file_path, table_name='logs'):
    """
    解析SQL文件中的INSERT语句，提取数据
    """
    print(f"正在解析SQL文件: {sql_file_path}")
    print("这可能需要几分钟时间，请耐心等待...")
    
    data = []
    
    with open(sql_file_path, 'r', encoding='utf-8') as f:
        buffer = ""
        insert_pattern = f"INSERT INTO `{table_name}` VALUES "
        
        for line_num, line in enumerate(f, 1):
            if line_num % 100000 == 0:
                print(f"  已处理 {line_num} 行...")
            
            # 跳过注释和空行
            if line.startswith('--') or line.startswith('/*') or line.startswith('*/') or not line.strip():
                continue
            
            buffer += line
            
            # 查找INSERT语句
            if insert_pattern in buffer:
                # 提取VALUES后面的内容
                try:
                    values_start = buffer.find(insert_pattern) + len(insert_pattern)
                    values_section = buffer[values_start:]
                    
                    # 使用正则表达式提取所有的值元组
                    pattern = r'\(([^)]+)\)'
                    matches = re.findall(pattern, values_section)
                    
                    for match in matches:
                        # 分割每个值
                        values = []
                        current = ""
                        in_quotes = False
                        quote_char = None
                        
                        for char in match:
                            if char in ('"', "'") and (not in_quotes or char == quote_char):
                                if in_quotes and char == quote_char:
                                    in_quotes = False
                                    quote_char = None
                                elif not in_quotes:
                                    in_quotes = True
                                    quote_char = char
                                current += char
                            elif char == ',' and not in_quotes:
                                values.append(current.strip().strip('"').strip("'"))
                                current = ""
                            else:
                                current += char
                        
                        if current:
                            values.append(current.strip().strip('"').strip("'"))
                        
                        if len(values) == 10:  # logs表有10列
                            data.append(values)
                    
                    buffer = ""
                    
                except Exception as e:
                    print(f"警告: 第{line_num}行解析出错: {e}")
                    buffer = ""
                    continue
    
    print(f"✓ SQL文件解析完成，共提取 {len(data)} 条记录")
    
    # 创建DataFrame
    columns = ['id', 'user_id', 'qid', 'docno', 'event_type', 
               'start_idx', 'end_idx', 'duration', 'pass_flag', 'timestamp']
    df = pd.DataFrame(data, columns=columns)
    
    # 转换数据类型
    df['id'] = pd.to_numeric(df['id'], errors='coerce')
    df['qid'] = pd.to_numeric(df['qid'], errors='coerce')
    df['start_idx'] = pd.to_numeric(df['start_idx'], errors='coerce')
    df['end_idx'] = pd.to_numeric(df['end_idx'], errors='coerce')
    df['duration'] = pd.to_numeric(df['duration'], errors='coerce')
    df['pass_flag'] = pd.to_numeric(df['pass_flag'], errors='coerce')
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
    
    return df

In [3]:
# ============================================================================
# 第一步：加载数据
# ============================================================================

print("Loading data...")
print()

# 读取logs数据 - 支持两种方式
# 方式1：从SQL文件直接解析
# 方式2：从CSV文件读取

USE_SQL = True  # 设置为True从SQL解析，False从CSV读取

if USE_SQL:
    try:
        sql_file = 'backup_qe_full_v1.sql'
        logs_df = parse_sql_insert_statements(sql_file, table_name='logs')
        print(f"✓ Logs loaded from SQL: {len(logs_df)} records")
    except FileNotFoundError:
        print(f"✗ {sql_file} not found. Trying CSV instead...")
        USE_SQL = False

if not USE_SQL:
    try:
        logs_df = pd.read_csv('logs.csv')
        print(f"✓ Logs loaded from CSV: {len(logs_df)} records")
    except FileNotFoundError:
        print("✗ Neither SQL file nor logs.csv found. Please prepare data first.")
        exit(1)

# 读取interleaving结果
try:
    interleaving_df = pd.read_csv('result_interleaved_with_text.csv')
    print(f"✓ Interleaving results loaded: {len(interleaving_df)} records")
except FileNotFoundError:
    print("✗ interleaving_results.csv not found.")
    exit(1)

print()

Loading data...

正在解析SQL文件: backup_qe_full_v1.sql
这可能需要几分钟时间，请耐心等待...
✓ SQL文件解析完成，共提取 33483 条记录
✓ Logs loaded from SQL: 33483 records
✓ Interleaving results loaded: 387 records



In [4]:
# ============================================================================
# 第二步：筛选目标用户
# ============================================================================

print("=" * 80)
print("User Filtering")
print("=" * 80)
print()

# 方法1：手动指定27个用户ID（如果你已经知道）
MANUAL_USERS = ['5b68c9eb87af310001584803',
                '55d51a6b8ce09000127d4821',
                '67181a55422d48f5727f08ec',
                '595cd88562431800019d691a',
                '610c66163e803a3564dfaf9c',
                '66435e2e5c5b5aceb2195908',
                '628f572a2d1304a34b9e1a37',
                '65de30b34b701c678a13c75b',
                '5f540fc665e3b6148c04f10e',
                '5f1088e27a43fe0c64ca5814',
                '66cc6eb364154a51d4d44eb1',
                '66cec5a3fdf1fe2c010e9971',
                '5c6e60b04e08ad00018cc995',
                '673e222bdb1a35935b3075f1',
                '659c067de92634488b349626',
                '5d68c8aa40524c00189e8ac2',
                '66fa1820329fa9bb88f2e3c6',
                '6012e262527c380ccb2a6c74',
                '668a9e53e7690b3b64d48e92',
                '5ca102f57114700016da340d',
                '5c6709ab926a5b0001eb29c1',
                '66d9cd06e207a93377f7820e',
                '6151e479b4ee16668948c7f2',
                '5d542c3a5af5900019bc80c4',
                '5f565a2b33c152071057bb04',
                '62d6d77df59450e545fb22f6',
                '63cedb764240865b7177c745',
                '66acbaf5c803ca1a1ef3f056',
                '669053bbebafdd430e770142',
                #'660b2346862ff2a3da9cfe3e',
                '6413a4a1a6e064ffb173f1ce',
                '67ddd4f85ab870ffd90cc888',
                '5bbb5e3f118c0600013c3202',
                '5d8b66f5d189bd001a378273',
                '5a143884bd04860001b01123',
                '607864ca34af93e712224f01',
                '66b73f9ba035651170b94c76',
                '5c81582c4c1a61000155584f',
                '5c4f5967aac8be0001716a65',
                '5c0676a61a20110001ea1a24',
                '615eda024a5f84f408625b4f',
                '682c3f62f2680e9a67130379',
                '6756d530acef7d25ac1164c8',
                '67a95d468d14bd6324b4bbba',
                '5ca5189dce3f920016239854',
                '5d8e154af2858200171fdb95',
                '5bdcf08f05ed110001ab2bbf',
                '66e4c4358de330a06b4d33f1',
                '668bf688d58206e61f399a88',
                '67cf0c70aa43b13b6168ecbf',
                '5f392fae9168cc472f0468f6',
                '6658c56f3d9230d99c3e6752',
                '63fbf0e3b18cc14adc0dbfb6',
                '647f87dfe5161fce9adc1b97',
                '5bb7abf3c190e20001e5c2c8']

# 方法2：自动选择（如果列表为空，设置为[]使用所有用户）
if len(MANUAL_USERS) == 0:
    print("No manual user list provided. Using all users in the dataset.")
    target_users = logs_df['user_id'].unique().tolist()
else:
    target_users = MANUAL_USERS
    print(f"Using manually specified user list: {len(target_users)} users")

print(f"Target users to analyze: {len(target_users)}")
print()

# 筛选目标用户的日志
print("Filtering logs for target users...")
original_count = len(logs_df)
logs_df = logs_df[logs_df['user_id'].isin(target_users)].copy()
filtered_count = len(logs_df)

print(f"✓ Filtered from {original_count} to {filtered_count} records")
print(f"✓ Logs from {logs_df['user_id'].nunique()} unique users")

if logs_df['user_id'].nunique() < len(target_users):
    missing_users = set(target_users) - set(logs_df['user_id'].unique())
    print(f"⚠️  Warning: {len(missing_users)} users have no logs in the dataset")
    if len(missing_users) <= 5:
        print(f"   Missing users: {list(missing_users)[:5]}")

print()


User Filtering

Using manually specified user list: 54 users
Target users to analyze: 54

Filtering logs for target users...
✓ Filtered from 33483 to 12929 records
✓ Logs from 54 unique users



In [5]:
# ============================================================================
# 第二步之二：时间筛选（过滤旧实验数据）⭐ 重要
# ============================================================================

print("Filtering by timestamp to remove old experiment data...")

# 方法1：指定当前实验的开始日期
EXPERIMENT_START_DATE = '2025-11-01'  # 修改为你的实验开始日期

# 方法2：或者直接设置为 None 跳过时间筛选
# EXPERIMENT_START_DATE = None

if EXPERIMENT_START_DATE is not None:
    # 转换为datetime
    logs_df['timestamp'] = pd.to_datetime(logs_df['timestamp'], errors='coerce')
    experiment_start = pd.to_datetime(EXPERIMENT_START_DATE)
    
    before_time_filter = len(logs_df)
    logs_df = logs_df[logs_df['timestamp'] >= experiment_start].copy()
    after_time_filter = len(logs_df)
    
    removed_count = before_time_filter - after_time_filter
    print(f"✓ Removed {removed_count} records before {EXPERIMENT_START_DATE}")
    print(f"✓ Remaining records: {after_time_filter}")
    
    # 显示时间范围
    if len(logs_df) > 0:
        min_time = logs_df['timestamp'].min()
        max_time = logs_df['timestamp'].max()
        print(f"✓ Data time range: {min_time} to {max_time}")
else:
    print("⚠️  Time filtering disabled. Using all historical data.")
    print("   This may include data from previous experiments!")

print()

Filtering by timestamp to remove old experiment data...
✓ Removed 537 records before 2025-11-01
✓ Remaining records: 12392
✓ Data time range: 2025-11-05 13:43:06 to 2025-11-13 19:45:20



In [6]:
# ============================================================================
# 第三步：数据预处理和标签映射
# ============================================================================

print("Data preprocessing...")
print()

# 显示origin_label的分布
print("Origin Label Distribution:")
label_counts = interleaving_df['origin_label'].value_counts()
print(label_counts)
print()

# 创建文档标签映射字典 (qid, docno) -> origin_label
doc_label_map = {}
for _, row in interleaving_df.iterrows():
    key = (str(row['qid']), str(row['docno']))
    doc_label_map[key] = row['origin_label']

print(f"✓ Created document label mapping: {len(doc_label_map)} documents")
print()

# 统计每种标签的文档数量
label_doc_counts = {
    'Both': len(interleaving_df[interleaving_df['origin_label'] == 'Both']),
    'A-Only': len(interleaving_df[interleaving_df['origin_label'] == 'A-Only']),
    'B-Only': len(interleaving_df[interleaving_df['origin_label'] == 'B-Only']),
    'Easy-Negative': len(interleaving_df[interleaving_df['origin_label'] == 'Easy-Negative'])
}

print("Document counts by label:")
for label, count in label_doc_counts.items():
    print(f"  {label}: {count} documents")
print()

Data preprocessing...

Origin Label Distribution:
origin_label
Both             166
A-Only           100
B-Only           100
Easy-Negative     21
Name: count, dtype: int64

✓ Created document label mapping: 387 documents

Document counts by label:
  Both: 166 documents
  A-Only: 100 documents
  B-Only: 100 documents
  Easy-Negative: 21 documents



In [7]:
# ============================================================================
# 第四步：分析所有事件统计
# ============================================================================

print("=" * 80)
print("Overall Statistics")
print("=" * 80)
print()

# 1. 总打开文档数量
open_doc_logs = logs_df[logs_df['event_type'] == 'OPEN_DOC']
total_open_docs = len(open_doc_logs)

# 2. 总passage选择数量（原始）
passage_selection_logs = logs_df[logs_df['event_type'] == 'PASSAGE_SELECTION']
total_raw_passages = len(passage_selection_logs)

# 3. 总取消数量（详细分类）
cancel_logs = logs_df[logs_df['event_type'] == 'CANCEL_DOC']
total_cancels = len(cancel_logs)

# 3a. Cancel without selection
cancel_without_selection = cancel_logs[
    (cancel_logs['start_idx'] == -1) & (cancel_logs['end_idx'] == -1)
]
total_cancel_without_selection = len(cancel_without_selection)

# 3b. Cancel after selection
cancel_after_selection = cancel_logs[
    (cancel_logs['start_idx'] != -1) | (cancel_logs['end_idx'] != -1)
]
total_cancel_after_selection = len(cancel_after_selection)

# 4. 取消去掉后的总passage选择数量
total_final_passages = total_raw_passages - total_cancel_after_selection

print(f"1. Total Documents Opened: {total_open_docs}")
print(f"2. Total Passage Selections (raw): {total_raw_passages}")
print(f"3. Total Cancellations: {total_cancels}")
print(f"   - Cancel without selection: {total_cancel_without_selection}")
print(f"   - Cancel after selection: {total_cancel_after_selection}")
print(f"4. Total Passage Selections (final): {total_final_passages}")
print()

Overall Statistics

1. Total Documents Opened: 5437
2. Total Passage Selections (raw): 4726
3. Total Cancellations: 761
   - Cancel without selection: 761
   - Cancel after selection: 0
4. Total Passage Selections (final): 4726



In [8]:
# ============================================================================
# 第五步：按origin_label统计passage选择
# ============================================================================

print("=" * 80)
print("Passage Selection by Origin Label")
print("=" * 80)
print()

# 为每个passage selection事件标注origin_label
passage_selections_with_label = []

for _, log in passage_selection_logs.iterrows():
    qid_str = str(int(log['qid'])) if pd.notna(log['qid']) else '0'
    docno_str = str(log['docno'])
    key = (qid_str, docno_str)
    
    label = doc_label_map.get(key, 'Unknown')
    
    passage_selections_with_label.append({
        'user_id': log['user_id'],
        'qid': qid_str,
        'docno': docno_str,
        'origin_label': label,
        'start_idx': log['start_idx'],
        'end_idx': log['end_idx'],
        'duration': log['duration']
    })

passage_df = pd.DataFrame(passage_selections_with_label)

# 统计每种标签的选择数量
label_selection_counts = passage_df['origin_label'].value_counts()

print("Passage selections by origin label (raw):")
for label in ['Both', 'A-Only', 'B-Only', 'Easy-Negative', 'Unknown']:
    count = label_selection_counts.get(label, 0)
    percentage = (count / total_raw_passages * 100) if total_raw_passages > 0 else 0
    
    # 计算每个文档的平均选择数
    doc_count = label_doc_counts.get(label, 0)
    if label == 'Unknown':
        doc_count = 1  # 避免除以0
    avg_per_doc = count / doc_count if doc_count > 0 else 0
    
    print(f"\n{label}:")
    print(f"  Total selections: {count}")
    print(f"  Percentage: {percentage:.2f}%")
    print(f"  Document count: {doc_count}")
    print(f"  Avg selections per document: {avg_per_doc:.2f}")

print()

# 处理cancel after selection，标注这些取消的origin_label
cancelled_passages_with_label = []

for _, log in cancel_after_selection.iterrows():
    qid_str = str(int(log['qid'])) if pd.notna(log['qid']) else '0'
    docno_str = str(log['docno'])
    key = (qid_str, docno_str)
    
    label = doc_label_map.get(key, 'Unknown')
    cancelled_passages_with_label.append({
        'qid': qid_str,
        'docno': docno_str,
        'origin_label': label
    })

cancelled_df = pd.DataFrame(cancelled_passages_with_label)

# 统计被取消的passage的label分布
if len(cancelled_df) > 0:
    print("Cancelled passages by origin label:")
    cancelled_label_counts = cancelled_df['origin_label'].value_counts()
    for label in ['Both', 'A-Only', 'B-Only', 'Easy-Negative', 'Unknown']:
        count = cancelled_label_counts.get(label, 0)
        print(f"  {label}: {count}")
    print()

# 计算最终的passage选择（减去取消的）
final_label_counts = {}
for label in ['Both', 'A-Only', 'B-Only', 'Easy-Negative', 'Unknown']:
    raw_count = label_selection_counts.get(label, 0)
    cancelled_count = cancelled_label_counts.get(label, 0) if len(cancelled_df) > 0 else 0
    final_count = raw_count - cancelled_count
    final_label_counts[label] = final_count

print("Passage selections by origin label (final, after removing cancellations):")
for label in ['Both', 'A-Only', 'B-Only', 'Easy-Negative', 'Unknown']:
    count = final_label_counts[label]
    percentage = (count / total_final_passages * 100) if total_final_passages > 0 else 0
    
    doc_count = label_doc_counts.get(label, 0)
    if label == 'Unknown':
        doc_count = 1
    avg_per_doc = count / doc_count if doc_count > 0 else 0
    
    print(f"\n{label}:")
    print(f"  Final selections: {count}")
    print(f"  Percentage: {percentage:.2f}%")
    print(f"  Document count: {doc_count}")
    print(f"  Avg selections per document: {avg_per_doc:.2f}")

print()

Passage Selection by Origin Label

Passage selections by origin label (raw):

Both:
  Total selections: 2507
  Percentage: 53.05%
  Document count: 166
  Avg selections per document: 15.10

A-Only:
  Total selections: 1063
  Percentage: 22.49%
  Document count: 100
  Avg selections per document: 10.63

B-Only:
  Total selections: 1123
  Percentage: 23.76%
  Document count: 100
  Avg selections per document: 11.23

Easy-Negative:
  Total selections: 33
  Percentage: 0.70%
  Document count: 21
  Avg selections per document: 1.57

Unknown:
  Total selections: 0
  Percentage: 0.00%
  Document count: 1
  Avg selections per document: 0.00

Passage selections by origin label (final, after removing cancellations):

Both:
  Final selections: 2507
  Percentage: 53.05%
  Document count: 166
  Avg selections per document: 15.10

A-Only:
  Final selections: 1063
  Percentage: 22.49%
  Document count: 100
  Avg selections per document: 10.63

B-Only:
  Final selections: 1123
  Percentage: 23.76%
  D

In [9]:
# ============================================================================
# 查询级别的系统偏好分析
# 目标：识别哪些查询受益于PRF，哪些受挫于PRF，哪些无所谓
# ============================================================================
print("=" * 80)
print("Query-Level System Preference Analysis")
print("=" * 80)
print()

# 获取所有查询ID
all_qids = passage_df['qid'].unique()
query_preference_results = []

for qid in all_qids:
    query_passages = passage_df[passage_df['qid'] == qid]
    
    # 统计A-Only (ColBERT-E2E) 和 B-Only (ColBERT-PRF) 的点击
    a_only_clicks = len(query_passages[query_passages['origin_label'] == 'A-Only'])
    b_only_clicks = len(query_passages[query_passages['origin_label'] == 'B-Only'])
    both_clicks = len(query_passages[query_passages['origin_label'] == 'Both'])
    
    # 统计该查询对应的各标签文档数量
    query_interleaving = interleaving_df[interleaving_df['qid'] == int(qid)]
    a_only_docs = len(query_interleaving[query_interleaving['origin_label'] == 'A-Only'])
    b_only_docs = len(query_interleaving[query_interleaving['origin_label'] == 'B-Only'])
    both_docs = len(query_interleaving[query_interleaving['origin_label'] == 'Both'])
    
    total_clicks = a_only_clicks + b_only_clicks + both_clicks
    
    if total_clicks == 0:
        continue
    
    # 计算偏好指标
    # 只看A-Only vs B-Only的对比（排除Both）
    discriminative_clicks = a_only_clicks + b_only_clicks
    
    if discriminative_clicks > 0:
        # B-Only占discriminative clicks的比例
        b_preference_ratio = b_only_clicks / discriminative_clicks
        
        # 分类这个查询
        # 阈值可以根据需要调整（0.6/0.4是一个合理的起点）
        if b_preference_ratio >= 0.6:
            preference = "PRF-Benefit"  # PRF明显更好
        elif b_preference_ratio <= 0.4:
            preference = "PRF-Hurt"     # PRF反而更差
        else:
            preference = "Neutral"      # 差不多
    else:
        # 只有Both点击，无法判断
        preference = "Insufficient-Data"
        b_preference_ratio = None
    
    query_preference_results.append({
        'qid': qid,
        'a_only_clicks': f"{a_only_clicks}({a_only_docs})",
        'b_only_clicks': f"{b_only_clicks}({b_only_docs})",
        'both_clicks': f"{both_clicks}({both_docs})",
        'total_clicks': total_clicks,
        'discriminative_clicks': discriminative_clicks,
        'b_preference_ratio': b_preference_ratio,
        'preference': preference
    })

# 转换为DataFrame
preference_df = pd.DataFrame(query_preference_results)
preference_df.to_csv('preference.csv', index=False)
preference_df

Query-Level System Preference Analysis



,qid,a_only_clicks,b_only_clicks,both_clicks,total_clicks,discriminative_clicks,b_preference_ratio,preference
0,104861,23(3),57(3),30(2),110,80,0.712500,PRF-Benefit
1,130510,12(2),30(2),93(5),135,42,0.714286,PRF-Benefit
2,131843,1(2),25(2),84(4),110,26,0.961538,PRF-Benefit
3,146187,27(2),8(2),85(5),120,35,0.228571,PRF-Hurt
4,148538,12(2),33(2),57(4),102,45,0.733333,PRF-Benefit
5,156493,17(1),19(1),113(6),149,36,0.527778,Neutral
6,1037798,3(3),40(3),55(3),98,43,0.930233,PRF-Benefit
7,1063750,20(3),6(3),30(2),56,26,0.230769,PRF-Hurt
8,1103812,8(1),4(1),70(7),82,12,0.333333,PRF-Hurt
9,1106007,30(2),40(2),74(4),144,70,0.571429,Neutral


In [10]:
# ============================================================================
# 输出统计结果
# ============================================================================

print(f"Total queries analyzed: {len(preference_df)}")
print()
print("Query Distribution by PRF Effect:")
print("-" * 80)

for pref in ['PRF-Benefit', 'PRF-Hurt', 'Neutral', 'Insufficient-Data']:
    subset = preference_df[preference_df['preference'] == pref]
    count = len(subset)
    percentage = count / len(preference_df) * 100
    
    print(f"\n{pref}:")
    print(f"  Count: {count} queries ({percentage:.2f}%)")
    
    if pref != 'Insufficient-Data' and count > 0:
        avg_ratio = subset['b_preference_ratio'].mean()
        avg_disc_clicks = subset['discriminative_clicks'].mean()
        print(f"  Average B-preference ratio: {avg_ratio:.3f}")
        print(f"  Average discriminative clicks: {avg_disc_clicks:.1f}")

Total queries analyzed: 43

Query Distribution by PRF Effect:
--------------------------------------------------------------------------------

PRF-Benefit:
  Count: 9 queries (20.93%)
  Average B-preference ratio: 0.779
  Average discriminative clicks: 47.1

PRF-Hurt:
  Count: 11 queries (25.58%)
  Average B-preference ratio: 0.280
  Average discriminative clicks: 36.3

Neutral:
  Count: 21 queries (48.84%)
  Average B-preference ratio: 0.500
  Average discriminative clicks: 64.9

Insufficient-Data:
  Count: 2 queries (4.65%)


In [11]:
# ============================================================================
# 详细输出每个类别的例子
# ============================================================================

print("\n" + "=" * 80)
print("Example Queries from Each Category:")
print("=" * 80)

for pref in ['PRF-Benefit', 'PRF-Hurt', 'Neutral']:
    subset = preference_df[preference_df['preference'] == pref].head(5)
    if len(subset) > 0:
        print(f"\n{pref} Examples:")
        print("-" * 80)
        for _, row in subset.iterrows():
            print(f"  QID: {row['qid']}")
            print(f"    A-Only: {row['a_only_clicks']}, B-Only: {row['b_only_clicks']}, Both: {row['both_clicks']}")
            print(f"    B-preference: {row['b_preference_ratio']:.3f}")


Example Queries from Each Category:

PRF-Benefit Examples:
--------------------------------------------------------------------------------
  QID: 104861
    A-Only: 23, B-Only: 57, Both: 30
    B-preference: 0.713
  QID: 130510
    A-Only: 12, B-Only: 30, Both: 93
    B-preference: 0.714
  QID: 131843
    A-Only: 1, B-Only: 25, Both: 84
    B-preference: 0.962
  QID: 148538
    A-Only: 12, B-Only: 33, Both: 57
    B-preference: 0.733
  QID: 1037798
    A-Only: 3, B-Only: 40, Both: 55
    B-preference: 0.930

PRF-Hurt Examples:
--------------------------------------------------------------------------------
  QID: 146187
    A-Only: 27, B-Only: 8, Both: 85
    B-preference: 0.229
  QID: 1063750
    A-Only: 20, B-Only: 6, Both: 30
    B-preference: 0.231
  QID: 1103812
    A-Only: 8, B-Only: 4, Both: 70
    B-preference: 0.333
  QID: 1112341
    A-Only: 50, B-Only: 6, Both: 1
    B-preference: 0.107
  QID: 1115776
    A-Only: 11, B-Only: 6, Both: 62
    B-preference: 0.353

Neutral Exa

In [12]:
# ============================================================================
# Selective PRF的潜在价值
# ============================================================================

print("\n" + "=" * 80)
print("Selective PRF Potential Value:")
print("=" * 80)

# 计算如果完美预测，能获得多少收益
total_a_only = preference_df['a_only_clicks'].sum()
total_b_only = preference_df['b_only_clicks'].sum()

# Oracle: 对每个查询选择更好的系统
oracle_benefit = 0
for _, row in preference_df.iterrows():
    oracle_benefit += max(row['a_only_clicks'], row['b_only_clicks'])

print(f"\nBaseline Performance:")
print(f"  Always use E2E (A-Only): {total_a_only} exclusive clicks")
print(f"  Always use PRF (B-Only): {total_b_only} exclusive clicks")
print(f"  Best static baseline: {max(total_a_only, total_b_only)} exclusive clicks")

print(f"\nOracle (Perfect Query-Level Selection):")
print(f"  Total exclusive clicks: {oracle_benefit}")
print(f"  Improvement over best baseline: {(oracle_benefit - max(total_a_only, total_b_only)) / max(total_a_only, total_b_only) * 100:.2f}%")

print(f"\nThis improvement represents the upper bound of what Selective PRF can achieve.")


Selective PRF Potential Value:

Baseline Performance:
  Always use E2E (A-Only): 1063 exclusive clicks
  Always use PRF (B-Only): 1123 exclusive clicks
  Best static baseline: 1123 exclusive clicks

Oracle (Perfect Query-Level Selection):
  Total exclusive clicks: 1314
  Improvement over best baseline: 17.01%

This improvement represents the upper bound of what Selective PRF can achieve.


In [12]:
# ============================================================================
# 第六步：计算ASR指标（Audit Success Rate）- 文档级别
# ============================================================================

print("=" * 80)
print("ASR Metrics (Audit Success Rate) - Document-Level Analysis")
print("=" * 80)
print()

# 获取所有查询ID
all_qids = passage_df['qid'].unique()

# 用于统计的变量
asr_least_docs = 0  # B-Only文档低于A-Only最小值的数量
asr_avg_docs = 0    # B-Only文档低于A-Only平均值的数量
asr_max_docs = 0    # B-Only文档低于A-Only最大值的数量

total_b_only_docs = 0  # 总的B-Only文档数（在有A-Only的查询中）

# 记录每个有问题的B-Only文档
problematic_docs = []

print(f"Analyzing {len(all_qids)} queries for B-Only document performance...")
print()

for qid in all_qids:
    # 获取该查询的所有passage选择
    query_passages = passage_df[passage_df['qid'] == qid]
    
    if len(query_passages) == 0:
        continue
    
    # 分别获取A-Only和B-Only文档的选择数
    a_only_docs = query_passages[query_passages['origin_label'] == 'A-Only']
    b_only_docs = query_passages[query_passages['origin_label'] == 'B-Only']
    
    # 如果该查询没有A-Only或B-Only文档，跳过
    if len(a_only_docs) == 0 or len(b_only_docs) == 0:
        continue
    
    # 计算每个A-Only文档的选择次数
    a_only_doc_selections = a_only_docs.groupby('docno').size()
    
    if len(a_only_doc_selections) == 0:
        continue
    
    # A-Only文档的统计值
    a_only_min = a_only_doc_selections.min()
    a_only_avg = a_only_doc_selections.mean()
    a_only_max = a_only_doc_selections.max()
    
    # 计算每个B-Only文档的选择次数
    b_only_doc_selections = b_only_docs.groupby('docno').size()
    
    # 对该查询中的每个B-Only文档进行检查
    for docno, b_selections in b_only_doc_selections.items():
        total_b_only_docs += 1
        
        issues = []
        
        # ASR-Least: B-Only文档选择数 < A-Only最小选择数
        if b_selections < a_only_min:
            asr_least_docs += 1
            issues.append('ASR-Least')
        
        # ASR-Avg: B-Only文档选择数 < A-Only平均选择数
        if b_selections < a_only_avg:
            asr_avg_docs += 1
            issues.append('ASR-Avg')
        
        # ASR-Max: B-Only文档选择数 < A-Only最大选择数
        if b_selections < a_only_max:
            asr_max_docs += 1
            issues.append('ASR-Max')
        
        # 如果有任何问题，记录这个文档
        if len(issues) > 0:
            problematic_docs.append({
                'qid': qid,
                'docno': docno,
                'b_only_selections': b_selections,
                'a_only_min': a_only_min,
                'a_only_avg': f'{a_only_avg:.2f}',
                'a_only_max': a_only_max,
                'issues': ', '.join(issues)
            })

# 计算ASR比率（文档级别）
asr_least_rate = (asr_least_docs / total_b_only_docs * 100) if total_b_only_docs > 0 else 0
asr_avg_rate = (asr_avg_docs / total_b_only_docs * 100) if total_b_only_docs > 0 else 0
asr_max_rate = (asr_max_docs / total_b_only_docs * 100) if total_b_only_docs > 0 else 0

print(f"Total B-Only documents analyzed: {total_b_only_docs}")
print()
print("ASR Metrics Results (Document-Level):")
print(f"  ASR-Least: {asr_least_docs} documents ({asr_least_rate:.2f}%)")
print(f"    → B-Only documents with selections < A-Only minimum")
print()
print(f"  ASR-Avg: {asr_avg_docs} documents ({asr_avg_rate:.2f}%)")
print(f"    → B-Only documents with selections < A-Only average")
print()
print(f"  ASR-Max: {asr_max_docs} documents ({asr_max_rate:.2f}%)")
print(f"    → B-Only documents with selections < A-Only maximum")
print()

ASR Metrics (Audit Success Rate) - Document-Level Analysis

Analyzing 22 queries for B-Only document performance...

Total B-Only documents analyzed: 45

ASR Metrics Results (Document-Level):
  ASR-Least: 8 documents (17.78%)
    → B-Only documents with selections < A-Only minimum

  ASR-Avg: 15 documents (33.33%)
    → B-Only documents with selections < A-Only average

  ASR-Max: 20 documents (44.44%)
    → B-Only documents with selections < A-Only maximum



In [13]:
# ============================================================================
# 第六步之二：查询级别汇总（Query-Level Summary）
# ============================================================================

print("=" * 80)
print("Query-Level Summary: Distribution of Problematic Documents")
print("=" * 80)
print()

# 统计有问题文档的查询分布
queries_with_least = set()
queries_with_avg = set()
queries_with_max = set()
queries_with_any_issue = set()

for doc in problematic_docs:
    qid = doc['qid']
    issues = doc['issues'].split(', ')
    
    if 'ASR-Least' in issues:
        queries_with_least.add(qid)
    if 'ASR-Avg' in issues:
        queries_with_avg.add(qid)
    if 'ASR-Max' in issues:
        queries_with_max.add(qid)
    
    queries_with_any_issue.add(qid)

# 统计总共有多少查询被分析（有A-Only和B-Only的查询）
queries_analyzed_for_asr = len(set(doc['qid'] for doc in problematic_docs + 
    [{'qid': qid} for qid in all_qids 
     if len(passage_df[(passage_df['qid'] == qid) & (passage_df['origin_label'] == 'A-Only')]) > 0
     and len(passage_df[(passage_df['qid'] == qid) & (passage_df['origin_label'] == 'B-Only')]) > 0]))

# 计算查询级别的比率
least_query_rate = (len(queries_with_least) / queries_analyzed_for_asr * 100) if queries_analyzed_for_asr > 0 else 0
avg_query_rate = (len(queries_with_avg) / queries_analyzed_for_asr * 100) if queries_analyzed_for_asr > 0 else 0
max_query_rate = (len(queries_with_max) / queries_analyzed_for_asr * 100) if queries_analyzed_for_asr > 0 else 0
any_issue_query_rate = (len(queries_with_any_issue) / queries_analyzed_for_asr * 100) if queries_analyzed_for_asr > 0 else 0

print(f"Total queries analyzed (with both A-Only and B-Only): {queries_analyzed_for_asr}")
print()
print("Queries containing problematic B-Only documents:")
print(f"  ASR-Least: {len(queries_with_least)} queries ({least_query_rate:.2f}%)")
print(f"    → Queries where at least one B-Only doc < A-Only minimum")
print()
print(f"  ASR-Avg: {len(queries_with_avg)} queries ({avg_query_rate:.2f}%)")
print(f"    → Queries where at least one B-Only doc < A-Only average")
print()
print(f"  ASR-Max: {len(queries_with_max)} queries ({max_query_rate:.2f}%)")
print(f"    → Queries where at least one B-Only doc < A-Only maximum")
print()
print(f"  Any Issue: {len(queries_with_any_issue)} queries ({any_issue_query_rate:.2f}%)")
print(f"    → Queries with at least one problematic B-Only document")
print()

# 统计每个查询的问题文档数量分布
query_problem_counts = {}
for doc in problematic_docs:
    qid = doc['qid']
    query_problem_counts[qid] = query_problem_counts.get(qid, 0) + 1

if len(query_problem_counts) > 0:
    print("Distribution of problematic documents per query:")
    problem_counts = list(query_problem_counts.values())
    print(f"  Min: {min(problem_counts)} documents")
    print(f"  Max: {max(problem_counts)} documents")
    print(f"  Average: {sum(problem_counts) / len(problem_counts):.2f} documents")
    print(f"  Median: {sorted(problem_counts)[len(problem_counts)//2]} documents")
    print()
    
    # 显示问题最严重的查询（前5个）
    top_problematic_queries = sorted(query_problem_counts.items(), key=lambda x: x[1], reverse=True)[:5]
    print("Top 5 queries with most problematic B-Only documents:")
    for i, (qid, count) in enumerate(top_problematic_queries, 1):
        # 计算该查询的 B-Only 总文档数
        total_b_in_query = len(passage_df[(passage_df['qid'] == qid) & 
                                          (passage_df['origin_label'] == 'B-Only')]['docno'].unique())
        problem_rate = (count / total_b_in_query * 100) if total_b_in_query > 0 else 0
        print(f"  {i}. Query {qid}: {count}/{total_b_in_query} B-Only docs problematic ({problem_rate:.1f}%)")

print()

Query-Level Summary: Distribution of Problematic Documents

Total queries analyzed (with both A-Only and B-Only): 20

Queries containing problematic B-Only documents:
  ASR-Least: 6 queries (30.00%)
    → Queries where at least one B-Only doc < A-Only minimum

  ASR-Avg: 9 queries (45.00%)
    → Queries where at least one B-Only doc < A-Only average

  ASR-Max: 11 queries (55.00%)
    → Queries where at least one B-Only doc < A-Only maximum

  Any Issue: 11 queries (55.00%)
    → Queries with at least one problematic B-Only document

Distribution of problematic documents per query:
  Min: 1 documents
  Max: 3 documents
  Average: 1.82 documents
  Median: 2 documents

Top 5 queries with most problematic B-Only documents:
  1. Query 1063750: 3/3 B-Only docs problematic (100.0%)
  2. Query 1133167: 3/3 B-Only docs problematic (100.0%)
  3. Query 146187: 2/2 B-Only docs problematic (100.0%)
  4. Query 1112341: 2/2 B-Only docs problematic (100.0%)
  5. Query 1114646: 2/2 B-Only docs problem